In [1]:
import gc
import string
from collections import defaultdict

import torch.cuda
from dotenv import load_dotenv

import pandas as pd
from tqdm.auto import tqdm

from datasets import Dataset
from huggingface_hub import login
from transformers.pipelines.pt_utils import KeyDataset
from transformers import pipeline, GenerationConfig, AutoTokenizer, AutoModelForCausalLM

from evaluation.evaluator import Evaluator
from evaluation.utils.prompt_templates import (
    MC_QUESTION_TEMPLATE,
    MATH_TEMPLATE,
    TRANSLATION_TEMPLATE
)

load_dotenv()
login()

In [2]:
dataset_configs = {
    "mmlu-pro": {
        "tasks": ["math"],
        "question_type": "mc",
        "category_column": "category",
    },
    "MATH": {
        "tasks": ["Algebra", "Intermediate Algebra"],
        "question_type": "math",
        "category_column": "subject",
    },
    "flores": {
        "tasks": ["cmn", "swh", "deu"],
        "question_type": "translation",
        "category_column": "iso_639_3"
    }
}

personas = [
    "no_persona", "helpful_persona", "base_persona",
    "static_short_persona", "static_long_persona",
    "dynamic_short_persona", "dynamic_long_persona",
    "beginner_teacher_persona", "intermediate_teacher_persona",
    "expert_teacher_persona"
]

In [3]:
def merge_and_write(
    dataframe: pd.DataFrame,
    subset_df: pd.DataFrame,
    file_name: str,
    id_column: str = "static_id"
):
    """Merges two dataframes.

    This function updates the entries in dataframe with the corresponding values in subset_df. Any
    entries in dataframe will be overwritten.

    Args:
        dataframe (pd.DataFrame): DataFrame object. The entries in this dataframe will be overwritten.
        subset_df (pd.DataFrame): Subset dataframe that will be used to update dataframe.
        file_name (str): File name of the written dataframe.
        id_column (str): Column to use as keys for the updates.
    """
    for col in subset_df.columns:
        if col not in dataframe.columns:
            dataframe[col] = None

    dataframe.set_index(id_column, inplace=True)
    subset_df = subset_df.set_index(id_column)

    # Any value in dataframe will be overwritten by subset_df
    dataframe.update(subset_df)
    dataframe.reset_index(inplace=True)

    # df_file = f"{drive_path}/{file_name}.parquet"
    df_file = f"data/{file_name}.parquet"
    dataframe.to_parquet(df_file)

In [ ]:
def fill_template(
    task_data: pd.Series,
    question_type: str
) -> str:
    """Chooses and fills a question template.

    The function first chooses the correct template based on the question type parameter
    and fills the template with the data from the task series.

    Args:
        task_data (pd.Series): Series containing the question and possibly answer options.
        question_type (str): String identifier of the question type.
            Can be 'mc', 'math' or 'translation'.

    Returns:
        str: The question template with filled placeholders.

    Raises:
        ValueError: If the question type is unknown.
    """
    if question_type == "mc":
        template = MC_QUESTION_TEMPLATE
    elif question_type == "math":
        template = MATH_TEMPLATE
    elif question_type == "translation":
        template = TRANSLATION_TEMPLATE
    else:
        raise ValueError(
            f"Invalid question type {question_type}. Must be 'mc', 'math' "
            f"or 'translation'.")

    prompt_kwargs = {"question": task_data["question"]}
    if question_type == "mc":
        letters = string.ascii_uppercase
        choices = "\n".join(
            f"({letters[i]}) {answer}"
            for i, answer in enumerate(task_data["answers"])
        )
        prompt_kwargs["choices"] = choices

    return template.format(**prompt_kwargs)

def prepare_prompt(batch, fn_tokenizer):
    prompts_list = []
    for system, user in zip(batch["persona"], batch["prompt"]):
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
        prompts_list.append(
            fn_tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        )
    return {"prompt": prompts_list}

In [ ]:
evaluator = Evaluator()

model_string = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_string, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(model_string)
pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    dtype="float16"
)
pipe.tokenizer.pad_token_id = pipe.tokenizer.eos_token_id

for ds_name, dataset_cfg in dataset_configs.items():
    print(f" === {ds_name} ===")
    _, dataset_df = evaluator.get_data(ds_name)
    dataset_df = dataset_df.copy()

    prompts = dataset_df.apply(
        fill_template, axis=1,
        question_type=dataset_cfg["question_type"])
    dataset_df.loc[:, "prompt"] = prompts

    # Sort the prompts by length to minimize padding overhead
    prompts_tokenized = pipe.tokenizer(prompts.to_list())
    dataset_df.loc[:, "length"] = [len(p) for p in prompts_tokenized["input_ids"]]
    dataset_df = dataset_df.sort_values(by="length")

    long_df = pd.melt(
        dataset_df,
        id_vars=["static_id", "prompt"],
        value_vars=personas,
        var_name="persona_col",
        value_name="persona")
    dataset_hf = Dataset.from_pandas(long_df)

    dataset_hf = dataset_hf.map(
        prepare_prompt,
        batched=True,
        fn_kwargs={"fn_tokenizer": tokenizer},
        num_proc=4
    )

    gen_cfg = GenerationConfig(
        max_new_tokens=256,
        pad_token_id=pipe.tokenizer.eos_token_id
    )
    for batch_size in [128, 64, 32, 16, 8, 4, 3, 2, 1]:
        print(f"Testing batch size {batch_size}")
        try:
            responses = defaultdict(list)
            # noinspection PyTypeChecker
            for static_id, persona, out in tqdm(
                zip(
                    long_df["static_id"],
                    long_df["persona_col"],
                    pipe(
                        KeyDataset(dataset_hf, "prompt"),
                        batch_size=batch_size, return_full_text=False,
                        generation_config=gen_cfg
                    )
                ), total=len(dataset_hf)
            ):
                responses["static_id"].append(static_id)
                responses["persona"].append(persona.replace("persona", "answer"))
                responses["completion"].append(out[0]["generated_text"])

            # pivot back from long to wide
            response_df = pd.DataFrame(responses)
            response_df = response_df.pivot(index="static_id", columns="persona", values="completion")
            response_df = response_df.reset_index()

            # merge and save
            merge_and_write(dataset_df, response_df, f"{ds_name}_model")
            break
        except torch.cuda.OutOfMemoryError:
            gc.collect()
            torch.cuda.empty_cache()
